# Interfacing with `pfvs`

> **What's in this notebook?** How to hand a jaxvacua geometry to the external [`pfvs`](https://github.com/natemacfadden/pfvs) PFV enumerator, cross-check the perturbatively-flat-vacuum (PFV) algebra between the two codes, and build a `PFVEFT` from the result.

`pfvs` is an **optional** dependency (`pip install jaxvacua[pfvs]`); `import jaxvacua` never requires it.

## Setup

In [ ]:
import time

import numpy as np
import jax.numpy as jnp
import jaxvacua as jvc

# Persistent XLA compilation cache: the reduced-EFT kernels (a Newton solve with
# a nested jacrev inside) take tens of seconds to COMPILE once.  Caching the
# compiled artefact on disk means that cost is paid once ever, not once per run
# of this notebook.  Numerics are unaffected -- only the compiled binary is
# cached.  Harmless to skip if the helper is unavailable.
try:
    from stringjax_tools import configure_compilation_cache
    configure_compilation_cache(min_compile_time_secs=1.0)
except Exception as _exc:                     # pragma: no cover
    print(f"(compilation cache not enabled: {_exc})")

# A standard LCS example: CP[1,1,1,6,9] degree-18 (h12 = 2).
# FluxVacuaFinder extends FluxEFT and adds the Newton vacuum solver
model = jvc.FluxVacuaFinder(h12=2, model_ID=1, maximum_degree=2,
                            prange=20)
M = jnp.array([-16.0, 50.0])
K = jnp.array([3.0, -4.0])
model

## The PFV algebra in jaxvacua

`model.pfv_data(M, K)` bundles the N-matrix, the flat direction $p = N^{-1}K$, the full flux vector and the PFV conditions (arXiv:2512.17095 §6.2).

In [ ]:
pfv = model.pfv_data(M, K)
print(pfv)
print("flat direction p =", np.asarray(pfv.p))
for name, (ok, _val) in pfv.check().items():
    if name != "p":
        print(f"  [{'OK' if bool(ok) else '--'}]  {name}")

## Bridging to `pfvs.CYData`

`lcs_tree.to_cydata_kwargs()` (needs no pfvs) maps the topology to `pfvs.CYData` arguments, and `lcs_tree.to_cydata()` builds the object.  Two conventions to note: pfvs' `h11` is the kappa dimension (= jaxvacua's `h12`), so pfvs' **`h21` is jaxvacua's `h11`**; and pfvs' `PFV(data, K, M)` takes **`K` before `M`**.

In [ ]:
from jaxvacua import has_pfvs
print("pfvs available:", has_pfvs())

kw = model.lcs_tree.to_cydata_kwargs()           # ungated (needs no pfvs)
print("pfvs h11 (= jaxvacua h12):", kw['kappa'].shape[0])
print("pfvs h21 (= jaxvacua h11):", kw['h21'])

## Cross-checking the p-vector and conditions

With `pfvs` installed, the two codes agree on the flat direction and the shared conditions (invertibility, $Np=K$, $K\cdot p = 0$).

In [ ]:
if has_pfvs():
    import pfvs
    data = model.lcs_tree.to_cydata()
    # pfvs uses exact flint arithmetic -> integer flux; arg order (data, K, M).
    pf = pfvs.PFV(data, np.asarray(K).astype(int), np.asarray(M).astype(int))
    print("pfvs     p :", np.asarray(pf.p, dtype=float))
    print("jaxvacua p :", np.asarray(model.pfv_p_vector(M, K), dtype=float))
    print("N invertible:", bool(pf.check_Ninvertible()),
          " N p == K:", bool(pf.check_NpK()),
          " K.p == 0:", bool(pf.check_orthogonality()))
else:
    print("Install jaxvacua[pfvs] to run the cross-check.")

## From fluxes to a `PFVEFT`

Finally, build the perturbatively-flat-vacuum effective theory: all moduli slaved to $\tau$ along $z = p\,\tau$, leaving a 1d theory in $\tau$. The leading-order **racetrack** gives an analytic estimate of the vacuum, which we then solve exactly.

Two knobs are easy to conflate, and they are **orthogonal**:

* **`mode`** — *how the moduli are slaved to $\tau$.* `"ansatz"` (default) uses the leading-order flat direction $z = p\,\tau$; `"eom"` Newton-solves the moduli F-terms $\partial_z W = 0$ at fixed $\tau$, capturing the exponentially small instanton correction to that direction.
* **`reduction`** — *how the $2\times2$ reduced $\tau$-Hessian is projected out of the full Hessian.* Four schemes: `"frozen"`, `"schur"`, `"autodiff"`, `"tangent"`.

This notebook's model is **LCS** (the flat direction is exactly linear), which makes it the cleanest place to see what each knob does; [12_freezer](../04_analysis_and_pipelines/12_freezer.ipynb) covers the coniLCS case where $z_{\rm cf}(\tau)$ is additionally nonlinear.

In [ ]:
from jaxvacua import PFVEFT

flux = model.pfv_to_flux(M, K)
rt = model.pfv_racetrack(M, K)          # leading-order 2-term racetrack
tau0 = complex(rt['tau0'])
print(f"racetrack estimate: tau0 = {tau0:.4f},  gs = {float(rt['gs']):.4f}")

### The vacuum: one solve, not a solve inside a solve

The obvious thing to write is a root-find on the reduced F-term `eft.DW_x_light`. Resist it: in `mode="eom"` *every* residual evaluation reconstructs the moduli by Newton iteration, so an outer solve in $\tau$ nests a solve inside each of its own steps.

jaxvacua's `newton_method_flux_vacua` instead solves the **full** stationarity $D_IW = 0$ — all moduli *and* $\tau$ — in a single pass. For a PFV that is the same vacuum: moduli F-flatness plus $\tau$ F-flatness *is* full stationarity. We then reconstruct the full real point **once** and hand it to every reduction below.

In [ ]:
eft = PFVEFT.from_fluxes(model, M, K, mode="eom")
p = model.pfv_p_vector(M, K)

# ONE solve for the full stationarity (moduli AND tau) -- no nesting.
moduli_vac, tau_vac, res_newton = model.newton_method_flux_vacua(
    p * tau0, tau0, flux, solver_mode="real", tol=1e-12, max_iters=300)
tau_vac = complex(tau_vac)
x_vac = jnp.array([tau_vac.real, tau_vac.imag])

# Reconstruct the full real point ONCE; every reduction below reuses it.
xf_vac = eft.full_real_point(x_vac, flux)

print(f"tau = {tau_vac:.6f},  gs = {1/tau_vac.imag:.4f}")
print(f"full    max|D_I W|    = "
      f"{float(jnp.max(jnp.abs(model.DW_x(xf_vac, flux)))):.2e}")
print(f"reduced |dV_x_light|  = "
      f"{float(jnp.max(jnp.abs(eft.dV_x_light(x_vac, flux)))):.2e}   (on-shell)")

# The full solve's moduli and the reduced reconstruction agree -- which is exactly
# why the outer solve never needed an inner Newton solve.
z_recon = eft.reconstruct_full_moduli(jnp.array([], dtype=complex), tau_vac, flux)
print(f"|z(full solve) - z(reconstruction)| = "
      f"{float(jnp.max(jnp.abs(jnp.asarray(moduli_vac) - z_recon))):.2e}")

### The four reductions

Write the light$\to$full map as $x_{\rm full}(\tau)$ and the full Hessian as $\nabla\nabla V$. The schemes differ in *which* reduced curvature they compute:

* **`"frozen"`** — $J^{T}(\nabla\nabla V)J$ with the **constant ansatz** tangent $J = (p, 1)$. Because the LCS ansatz is linear, this is *exactly* the `"autodiff"` answer **in `mode="ansatz"`** (there $\partial^2 x_{\rm full}/\partial\tau^2 = 0$). In `mode="eom"` it keeps that same constant tangent while the point moves onto the eom slaving, so it no longer matches — on an exponentially small mass the mismatch is large.
* **`"schur"`** — the Schur complement $H_{\ell\ell} - H_{\ell h}H_{hh}^{-1}H_{h\ell}$, which integrates the moduli out **at their V-minimum** ($\partial_z V = 0$). This is the right reduction when a genuinely *heavy* modulus is integrated out at the bottom of its potential — the `ConifoldFreezer` case. A PFV is different: its moduli are slaved along the **F-flat** direction ($\partial_z W = 0$), which is *not* a V-valley, so `schur` answers a different question here.
* **`"autodiff"`** — `jax.hessian` of $V(x_{\rm full}(\tau))$ straight through the solve. Exact everywhere, most expensive.
* **`"tangent"`** — $J^{T}(\nabla\nabla V)J$ with the **true on-shell** tangent $J = \partial x_{\rm full}/\partial\tau$, obtained from one implicit-function-theorem linear solve on the `dDW_x` moduli block. Same physical value as `"autodiff"` at a vacuum, at first-order cost.

The **ground truth** is a finite difference of the reduced potential `V_x_light` — that *is* the racetrack potential in $\tau$ with the moduli slaved, so its curvature is the racetrack $\tau$-mass.

In [ ]:
import numpy as np

def fd_hessian(f, x, h=1e-4):
    # Central 2nd-order finite-difference Hessian (ground truth).
    n = x.size
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            xpp, xpm = x.copy(), x.copy()
            xmp, xmm = x.copy(), x.copy()
            xpp[i] += h; xpp[j] += h
            xpm[i] += h; xpm[j] -= h
            xmp[i] -= h; xmp[j] += h
            xmm[i] -= h; xmm[j] -= h
            H[i, j] = (f(xpp) - f(xpm) - f(xmp) + f(xmm)) / (4 * h * h)
    return 0.5 * (H + H.T)

V = lambda x: float(eft.V_x_light(jnp.asarray(x), flux))
H_fd = fd_hessian(V, np.asarray(x_vac, dtype=float))
ev_fd = np.linalg.eigvalsh(H_fd)
print("finite-difference (ground truth):", ev_fd)

In [ ]:
# Reconstruct the on-shell point ONCE and reuse it for every reduction.  Without
# this each call would repeat the heavy Newton solve -- the dominant cost -- even
# though the vacuum was already solved above.
# xf_vac already computed in the solve cell above -- reused, not re-solved

# Separate the ONE-OFF compile from the per-call cost, otherwise the first
# reduction absorbs the compile of kernels that all the others then reuse, and
# the table says the opposite of the truth.
t0 = time.time()
eft.ddV_x_light(x_vac, flux, reduction="tangent", x_full=xf_vac).block_until_ready()
t_compile = time.time() - t0
print(f"one-off XLA compile (shared by every reduction): {t_compile:.1f} s")
print("  -> with the persistent cache enabled above, later runs reuse it\n")

print(f"{'reduction':10s} {'eigenvalues':34s} {'rel. err vs FD':>14s}  {'warm':>9s}")
for red in ("frozen", "schur", "tangent", "autodiff"):
    # autodiff differentiates *through* the heavy solve, so it cannot reuse xf_vac
    kw = {} if red == "autodiff" else {"x_full": xf_vac}
    H = np.asarray(eft.ddV_x_light(x_vac, flux, reduction=red, **kw))   # warm up
    reps = 1 if red == "autodiff" else 3
    t0 = time.time()
    for _ in range(reps):
        eft.ddV_x_light(x_vac, flux, reduction=red, **kw)
    dt = (time.time() - t0) / reps
    ev = np.linalg.eigvalsh(0.5 * (H + H.T))
    rel = np.linalg.norm(H - H_fd) / np.linalg.norm(H_fd)
    unit = f"{dt*1e3:7.2f} ms" if dt < 1 else f"{dt:7.2f} s "
    print(f"{red:10s} {np.array2string(ev, precision=4):34s} {rel:14.2e}  {unit}")

Two practical points the table above makes concrete.

**Reconstruct once.** The heavy solve dominates: `full_real_point` returns the on-shell point, and passing it as `x_full` lets every reduction reuse it. `autodiff` is the exception — it differentiates *through* the solve, so it must redo it, which is why it stays milliseconds-to-seconds while the others are well under a millisecond.

**Compile is not run time.** These kernels are JIT-compiled; the first call pays a one-off XLA compile (tens of seconds, dominated by the Newton solve with its nested `jacrev`), and every later call is sub-millisecond. The setup cell enables JAX's persistent on-disk cache so that cost is paid once ever rather than once per notebook run. If you are timing anything in JAX, always warm up first — otherwise you are measuring the compiler.

Two things to read off. `"tangent"` and `"autodiff"` reproduce the finite-difference curvature (and each other); `"frozen"` is orders of magnitude too large; `"schur"` is a few $\times$ **too small** — systematically, not noisily.

The `"frozen"` failure is specific to `mode="eom"`. Rebuilding the same EFT in `mode="ansatz"`, where the light$\to$full map really is the linear ansatz, `frozen` and `autodiff` agree to machine precision — the claim "for LCS the linear flat direction makes `frozen` exact" is true, but *only* for the ansatz map:

In [ ]:
eft_ansatz = PFVEFT.from_fluxes(model, M, K, mode="ansatz")
x_a = jnp.array([tau0.real, tau0.imag])
H_fr = np.asarray(eft_ansatz.ddV_x_light(x_a, flux, reduction="frozen"))
H_ad = np.asarray(eft_ansatz.ddV_x_light(x_a, flux, reduction="autodiff"))
print("mode='ansatz'  frozen  :", np.linalg.eigvalsh(0.5 * (H_fr + H_fr.T)))
print("mode='ansatz'  autodiff:", np.linalg.eigvalsh(0.5 * (H_ad + H_ad.T)))
print("relative difference    :",
      np.linalg.norm(H_fr - H_ad) / np.linalg.norm(H_ad))

### Physical masses, and the default

The physical masses come from the generalised eigenproblem $H_{\rm eff}v = \lambda K_{\rm eff}v$ with the reduced Kähler metric, via `light_mass_spectrum` ($m^2 = \tfrac12\lambda$ in the supergravity normalisation). The reduction difference is **not** washed out by the metric pairing, so `light_mass_spectrum` defaults to `"tangent"` on a `PFVEFT` (the racetrack mass), while `ConifoldFreezer` keeps the `"schur"` default that is correct for a genuinely heavy modulus.

In [ ]:
for red in (None, "tangent", "autodiff", "schur", "frozen"):
    kw = {} if red is None else {"reduction": red}
    s = eft.light_mass_spectrum(x_vac, flux, dw_tol=1e-2, **kw)
    label = "default" if red is None else red
    print(f"{label:9s} -> reduction={s.reduction:9s}  masses={s.masses}  "
          f"stable={s.stable}")

**Which mode + reduction should I use?**

| situation | choice | why |
| --- | --- | --- |
| **PFV masses (LCS or coniLCS)** | **`mode="eom"` + `reduction="tangent"`** | on-shell point, F-flat slaving — the racetrack mass, at first-order cost (the `PFVEFT` default) |
| need the exact off-shell Hessian | `mode="eom"` + `reduction="autodiff"` | full `jax.hessian` through the solve; same value at a vacuum, much slower |
| quick look on the ansatz | `mode="ansatz"` + `reduction="frozen"` | exact for the *linear* LCS ansatz map, and cheap — but off-shell for coniLCS |
| a genuinely heavy modulus (`ConifoldFreezer`) | `reduction="schur"` | the modulus really is integrated out at its V-minimum |
| — avoid — | `mode="eom"` + `reduction="frozen"` | the frozen tangent is the *ansatz* one; it does not see the eom slaving |

`schur` is not "wrong": it is an exact V-minimum reduction, which simply is not the flat-direction curvature a PFV mass asks for. For the coniLCS version of this discussion — where the ansatz is additionally *off-shell* — see [12_freezer](../04_analysis_and_pipelines/12_freezer.ipynb).

## Take-aways

- `lcs_tree.to_cydata()` / `lcs_tree.to_cydata_kwargs()` bridge jaxvacua geometries into `pfvs`; the p-vector and shared conditions match.
- Mind the conventions: pfvs `h21` = jaxvacua `h11`, and `PFV(data, K, M)`.
- `pfvs` is optional — `has_pfvs()` gates any pfvs-dependent step.
- `mode` and `reduction` are orthogonal: `mode` sets *where* the point is (ansatz vs on-shell), `reduction` sets *which* reduced curvature is computed.
- For a PFV mass use `mode="eom"` + `reduction="tangent"` (the default) — it matches the finite-difference racetrack curvature; `frozen` is exact only on the linear ansatz map, and `schur` computes the V-minimum reduction instead.
